In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

In [ ]:
# File paths
DATASET_ROOT = "/root/.cache/kagglehub/datasets/kuanghangdong/bitvehicle/versions/1"
IMAGE_FOLDER = os.path.join(DATASET_ROOT, "BITVehicle")  # images live in BITVehicle subfolder
CSV_FILE = os.path.join(IMAGE_FOLDER, "VehicleInfo.csv")
IMG_SIZE = (64, 64)
TEST_SIZE = 0.2
MODEL_DIR = os.getcwd()  # Directory where model files will be saved


df = pd.read_csv(CSV_FILE)

print("Loaded:", df.shape)
print(df.head())


In [ ]:
# Visualize class distribution to identify imbalance
class_counts = df["category"].value_counts()
print("Class distribution:")
print(class_counts)
print(f"\nImbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x")

plt.figure(figsize=(8, 4))
class_counts.plot(kind="bar")
plt.title("Class Distribution (Before Balancing)")
plt.xlabel("Vehicle Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Extract features: HOG (shape) + HSV color histogram (color distribution)
# Combining both gives the model shape AND appearance cues, helping separate
# visually similar classes like Microbus/Minivan/SUV.
features = []
labels = []

print("Processing images...")

for _, row in df.iterrows():
    img_name = row["name"]
    label = row["category"]

    left   = int(row["left"])
    top    = int(row["top"])
    right  = int(row["right"])
    bottom = int(row["bottom"])

    img_path = os.path.join(IMAGE_FOLDER, img_name)
    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    crop = img[top:bottom, left:right]
    if crop.size == 0:
        continue

    crop = cv2.resize(crop, IMG_SIZE)

    # --- HOG on grayscale (shape/edge features) ---
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    # --- HSV color histogram (color distribution features) ---
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist_h = np.histogram(hsv[:, :, 0], bins=16, range=(0, 180))[0].astype(float)
    hist_s = np.histogram(hsv[:, :, 1], bins=16, range=(0, 256))[0].astype(float)
    hist_v = np.histogram(hsv[:, :, 2], bins=16, range=(0, 256))[0].astype(float)
    # Normalize so brightness differences don't dominate
    for hist in (hist_h, hist_s, hist_v):
        total = hist.sum()
        if total > 0:
            hist /= total
    color_features = np.concatenate([hist_h, hist_s, hist_v])  # 48 values

    features.append(np.concatenate([hog_features, color_features]))
    labels.append(label)

features = np.array(features)
labels = np.array(labels)

print(f"Feature shape: {features.shape}  "
      f"(HOG: {len(hog_features)}, color: {len(color_features)})")

In [ ]:
# Encode lables
le = LabelEncoder()
y = le.fit_transform(labels)

# Split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(
    features, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=42
)

In [ ]:
# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# SMOTE is now applied inside the GridSearchCV pipeline (see next cell).
# Applying it here before CV caused data leakage: synthetic validation samples
# were generated using neighbors from the training fold.
print("SMOTE will be applied per-fold inside the pipeline.")

In [ ]:
# GridSearchCV with StratifiedKFold cross-validation
# - SMOTE is inside the pipeline so it only sees each fold's training split (no leakage)
# - probability=True removed: it adds Platt calibration (an internal 5-fold CV per fit),
#   roughly doubling training time. Add it back only if you need predict_proba().
# - n_jobs=-1: runs folds/params in parallel

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'svm__C':     [1, 10, 50],
    'svm__gamma': ['scale', 0.01, 0.001],
}

pipe = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('svm',   SVC(kernel='rbf', class_weight='balanced')),
])

print("Running GridSearchCV (5-fold, macro F1 scoring)...")
grid_search = GridSearchCV(
    pipe,
    param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2,
    refit=True,
)
grid_search.fit(X_train, y_train)

svm = grid_search.best_estimator_
print("\nBest params:", grid_search.best_params_)
print(f"Best CV macro F1: {grid_search.best_score_:.4f}")

In [ ]:
# Per-fold scores for the best parameter combination, read directly from
# grid_search.cv_results_ — no second CV run needed.
best_idx = grid_search.best_index_
cv_scores = np.array([
    grid_search.cv_results_[f'split{i}_test_score'][best_idx]
    for i in range(5)
])

print("Per-fold macro F1 scores (best params):")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.4f}")
print(f"\nMean: {cv_scores.mean():.4f}  |  Std: {cv_scores.std():.4f}")

plt.figure(figsize=(7, 4))
plt.bar(range(1, len(cv_scores) + 1), cv_scores, color='steelblue', alpha=0.8)
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Mean = {cv_scores.mean():.3f}')
plt.xlabel("Fold")
plt.ylabel("Macro F1")
plt.title("5-Fold Cross-Validation — Macro F1 per Fold")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate SVM
y_pred = svm.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=le.classes_,
            yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Save model
import joblib

joblib.dump(svm, os.path.join(MODEL_DIR, "svm_model.pkl"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.pkl"))
joblib.dump(le, os.path.join(MODEL_DIR, "label_encoder.pkl"))
print("Models saved to:", MODEL_DIR)
